In [0]:
from pyspark.sql import functions as F
from datetime import datetime

# DEV MODE: Using CSV files from volume (PostgreSQL secrets not configured)
seed_path = "/Volumes/workspace/bronze/data_source/"

def ingest_from_csv(csv_name: str, target: str, mode: str = "overwrite"):
    df = (spark.read
          .option("header", "true")
          .option("inferSchema", "true")
          .csv(f"{seed_path}{csv_name}.csv"))
    df = df.withColumn("_ingested_at", F.lit(datetime.now()))
    
    (df.write
     .format("delta")
     .mode(mode)
     .saveAsTable(target))
    
    count = df.count()
    print(f"Bronze {target}: {count:,} rows")

# Ingest available tables from CSV
ingest_from_csv("events", "bronze.events", mode="overwrite")
ingest_from_csv("venues", "bronze.venues", mode="overwrite")
ingest_from_csv("event_tags", "bronze.event_tags", mode="overwrite")
ingest_from_csv("event_tag_map", "bronze.event_tags_map", mode="overwrite")